Just in case the following code doesn't work, consider downgrading your Numpy to < 1.16.0 and Scipy to < 1.1.0 
(Note this is just for this evoMPS engine implementation of Ash Milsted's code. For the other ipynb files, you would need the latest versions)

In [1]:
import math as ma
import numpy as np
import evoMPS.tdvp_uniform as tdvp
import evoMPS.dynamics as dy

S = 1
block_length = 1
bond_dim = 32
Jx = 1.00; Jy = 1.00; Jz = 1.00
tol = 1E-7
step = 0.04
max_steps = 10000
zero_tol = 1E-20

Sx = ma.sqrt(0.5) * np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=complex)
Sy = ma.sqrt(0.5) * 1.j * np.array([[0, -1, 0], [1, 0, -1], [0, 1, 0]], dtype=complex)
Sz = np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]], dtype=complex)
qn = 3

def get_ham(Jx, Jy, Jz):
    h = (Jx * np.kron(Sx, Sx) + Jy * np.kron(Sy, Sy) + Jz * np.kron(Sz, Sz)).reshape(qn, qn, qn, qn)
    return h.real

s = tdvp.EvoMPS_TDVP_Uniform(bond_dim, qn, get_ham(Jx, Jy, Jz), L=block_length)
s.symm_gauge = True
s.zero_tol = zero_tol

grnd_fname = f"heis_af_uni_L{block_length}_D{bond_dim}_q{qn}_S{S}_Jx{Jx}_Jy{Jy}_Jz{Jz}_s{tol}_dtau{step}_ground.npy"

print("--- 1. Calculating Infinite Uniform Ground State ---")
def cbf(s, i, **kwargs):
    if i % 100 == 0:
        print(f"Step {i:4d} | Energy: {s.h_expect.real:.8f} | eta: {s.eta.real:.2e}")
        
s = dy.find_ground(s, tol=tol, h_init=step, cb_func=cbf, max_itr=max_steps)
s.save_state(grnd_fname)
print(f"--- Ground state saved to {grnd_fname} ---")

INFO: No c versions of epsilon maps. Compile extension modules for a boost at low bond-dimensions.
--- 1. Calculating Infinite Uniform Ground State ---
Step    0 | Energy: 0.03928833 | eta: 9.56e-01


INFO:evoMPS.tdvp_uniform:CG: RESET due to bad search direction.


Step  100 | Energy: -1.40148362 | eta: 9.70e-05


INFO:evoMPS.tdvp_uniform:CG: RESET due to bad search direction.


--- Ground state saved to heis_af_uni_L1_D32_q3_S1_Jx1.0_Jy1.0_Jz1.0_s1e-07_dtau0.04_ground.npy ---


In [3]:
import time
# --- MONKEY PATCH FOR PYTHON 3.8+ ---
if not hasattr(time, 'clock'):
    time.clock = time.perf_counter
# ------------------------------------

import evoMPS.tdvp_sandwich as sw
import numpy as np
import matplotlib.pyplot as plt
import math as ma
import evoMPS.tdvp_uniform as tdvp

# Recreate the exact filename to load the saved state
S = 1; block_length = 1; bond_dim = 32; qn = 3
Jx = 1.00; Jy = 1.00; Jz = 1.00
tol = 1E-7; step = 0.04
grnd_fname = f"heis_af_uni_L{block_length}_D{bond_dim}_q{qn}_S{S}_Jx{Jx}_Jy{Jy}_Jz{Jz}_s{tol}_dtau{step}_ground.npy"

Sx = ma.sqrt(0.5) * np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=complex)
Sy = ma.sqrt(0.5) * 1.j * np.array([[0, -1, 0], [1, 0, -1], [0, 1, 0]], dtype=complex)
Sz = np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]], dtype=complex)

def get_ham(Jx, Jy, Jz):
    return (Jx * np.kron(Sx, Sx) + Jy * np.kron(Sy, Sy) + Jz * np.kron(Sz, Sz)).reshape(qn, qn, qn, qn).real

# Load the uniform ground state
hu_s = tdvp.EvoMPS_TDVP_Uniform(bond_dim, qn, get_ham(Jx, Jy, Jz), L=block_length)
with open(grnd_fname, 'rb') as f:
    hu_s.load_state(f)

N = 192    
dt = 0.01  
steps = 1500 # This evolves to t = 15.0

print(f"--- 2. Initializing sMPS Window (N={N}) ---")
sim = sw.EvoMPS_TDVP_Sandwich(N, hu_s)

Sp = Sx + 1.j * Sy
Sm = Sx - 1.j * Sy
mid = N // 2

print("--- 3. Injecting Excitations at m = +/- 10, +/- 20 ---")
sim.apply_op_1s(Sp, mid - 15 - 5)
sim.apply_op_1s(Sm, mid - 15 + 5)
sim.apply_op_1s(Sm, mid + 15 - 5)
sim.apply_op_1s(Sp, mid + 15 + 5)

print(f"--- 4. Running Real-Time TDVP Evolution (dt={dt}, steps={steps}) ---")
print("    (This will take a few minutes. Watch the terminal output!)")
# Multiplying dt by 1.j triggers real-time evolution
op, en, S_ent, OL = sw.go(sim, dt * 1.j, steps, RK4=True, op=Sz, op_every=10,
                          autogrow=True, autogrow_amount=4 // hu_s.L)

print("--- 5. Plotting Figure 2 ---")
op = np.array(op)
plt.figure(figsize=(10, 6))
# Render the heatmap
plt.imshow(op, origin="lower", interpolation="none", cmap="jet",
           aspect="auto", extent=(-N//2, N//2, 0, steps * dt), vmin=-0.5, vmax=0.5)
plt.xlabel('Site $n$')
plt.ylabel('Time $t (\hbar/J)$')
plt.title('Real-Time Scattering of Entangled Excitations (evoMPS)')
cb = plt.colorbar()
cb.set_label(r'$\langle S_n^z \rangle$')
plt.show()

--- 2. Initializing sMPS Window (N=192) ---
Bulk eta:  [9.54794774e-15+0.j]
--- 3. Injecting Excitations at m = +/- 10, +/- 20 ---
--- 4. Running Real-Time TDVP Evolution (dt=0.01, steps=1500) ---
    (This will take a few minutes. Watch the terminal output!)
Step	CPU	t	eta	E_nonuniform	E - E_prev	grown_left	grown_right

0	1.2	0.0	0	5.5855476129889325	5.5855476129889325	0	0
1	17.8	0.01	1.873747438467475	5.585544121205999	-3.4917829339065065e-06	0	0
2	33.9	0.02	1.8738539321714498	5.58557987040632	3.574920032178852e-05	0	0
3	45.5	0.03	1.8739174535492926	5.585579696486263	-1.7392005702276947e-07	0	0
4	56.0	0.04	1.8739371781567893	5.585579615390429	-8.109583404802834e-08	0	0
5	64.4	0.05	1.8739453676891917	5.585579593138448	-2.2251981590670766e-08	0	0
6	80.1	0.06	1.8739486837690007	5.585579584207608	-8.93084006747813e-09	0	0
7	92.1	0.07	1.8739501688676259	5.585579581520733	-2.6868747227126732e-09	0	0
8	109.4	0.08	1.873950919570758	5.585579583432946	1.9122126104775816e-09	0	0
9	125.2	0.09	1.

AttributeError: 'RcParams' object has no attribute '_get'

In [4]:
import numpy as np

# This grabs the matrix from your RAM and locks it to your hard drive
np.save("scattering_data.npy", op)
print("Data safely secured to hard drive!")

Data safely secured to hard drive!


In [5]:
import matplotlib
# Force matplotlib to draw silently to a file instead of the screen
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# Re-establish the variables just in case
N = 192
dt = 0.01
steps = 1500

plt.figure(figsize=(10, 6))
# Render the heatmap
plt.imshow(op, origin="lower", interpolation="none", cmap="jet",
           aspect="auto", extent=(-N//2, N//2, 0, steps * dt), vmin=-0.5, vmax=0.5)
plt.xlabel('Site $n$')
plt.ylabel('Time $t (\hbar/J)$')
plt.title('Real-Time Scattering of Entangled Excitations (evoMPS)')
cb = plt.colorbar()
cb.set_label(r'$\langle S_n^z \rangle$')

# Save the exact thesis-quality image to your folder
plt.savefig("scattering_plot_Figure2.png", dpi=300, bbox_inches='tight')
print("--- Plot saved as scattering_plot_Figure2.png in your folder! ---")

--- Plot saved as scattering_plot_Figure2.png in your folder! ---
